# Orchestration Pipeline Lab
## Enterprise AI Systems with Azure OpenAI and LangChain

This lab covers how enterprise AI systems are composed from individual LLM calls into coordinated, multi-step pipelines. Each section addresses a distinct orchestration concern: how work flows, how the model reasons through complexity, how outputs are validated before being trusted, how pipelines branch based on content, and how all of this connects to no-code platforms like Copilot Studio and Power Platform.

The progression is deliberate. You will first understand how to orchestrate steps in sequence and in parallel, then layer in reasoning strategies that make those steps more reliable, then add validation so the pipeline can catch and recover from bad outputs, then introduce branching so the pipeline can handle heterogeneous inputs intelligently. The final sections show how to expose these pipelines to non-technical users through Microsoft Copilot Studio and Power Platform, closing the loop between engineering and business.

Every section follows the same structure: the concept is explained in plain language, the code demonstrates it working against a real Azure OpenAI endpoint, and a closing note explains when you would reach for this pattern in your own systems.

## Lab Contents

- Section 0: Environment Setup
- Section 1: Workflow Orchestration
- Section 2: Intermediate Reasoning (CoT, ReAct, Self-Reflection)
- Section 3: Validation Prompts
- Section 4: Branching
- Section 5: Copilot Studio and No-Code Agentic Systems
- Section 6: Copilot Studio and Power Platform with Dataverse
- Section 7: End-to-End Orchestration Pipeline

## Section 0: Environment Setup

This notebook is designed to be standalone. Run the cell below once to install dependencies, then configure your Azure OpenAI credentials. The `get_llm` factory defined here is reused in every subsequent section, so it is important to run this section before proceeding.

You will need an Azure OpenAI resource with at least one deployed chat model (gpt-4o recommended) It is available through the Azure Portal under your OpenAI resource's Model Deployments blade.

In [1]:
%pip install -q langchain langchain-openai langchain-core langchain-community \
    langchain-text-splitters langgraph pydantic tiktoken faiss-cpu python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set your Azure OpenAI credentials here or via a .env file
os.environ.setdefault("AZURE_OPENAI_API_KEY",        "")
os.environ.setdefault("AZURE_OPENAI_ENDPOINT",       "https://<TBD>.cognitiveservices.azure.com")
os.environ.setdefault("AZURE_OPENAI_DEPLOYMENT_NAME","gpt-4o")
os.environ.setdefault("AZURE_OPENAI_API_VERSION",    "2024-12-01-preview")

AZURE_API_KEY     = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_ENDPOINT    = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_DEPLOYMENT  = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]
AZURE_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]

print("Credentials loaded.")
print(f"  Endpoint  : {AZURE_ENDPOINT}")
print(f"  Deployment: {AZURE_DEPLOYMENT}")

Credentials loaded.
  Endpoint  : https://<TBD>.cognitiveservices.azure.com
  Deployment: gpt-4o


In [3]:
from langchain_openai import AzureChatOpenAI

def get_llm(temperature: float = 0.0, max_tokens: int = 1024) -> AzureChatOpenAI:
    """
    Central factory for AzureChatOpenAI instances.
    All sections use this function so credential changes only need to happen once.

    temperature=0.0 produces deterministic outputs, suitable for validation and classification.
    temperature=0.7 produces more varied outputs, suitable for generation and reasoning.
    """
    return AzureChatOpenAI(
        azure_deployment=AZURE_DEPLOYMENT,
        azure_endpoint=AZURE_ENDPOINT,
        api_key=AZURE_API_KEY,
        api_version=AZURE_API_VERSION,
        temperature=temperature,
        max_tokens=max_tokens,
    )

# Smoke test
llm = get_llm()
resp = llm.invoke("Reply with exactly: Orchestration Lab Ready")
print(resp.content)

Orchestration Lab Ready


## Section 1: Workflow Orchestration

### Concept

Workflow orchestration is the practice of coordinating multiple AI steps into a predictable, repeatable sequence. A single LLM call is rarely enough for a real business task. Consider a contract review process: you need to extract key clauses, classify the contract type, assess risk, generate a summary, and route the document to the right reviewer. Each of those steps is a distinct operation, and the output of one feeds the next.

LangChain's Expression Language (LCEL) uses the pipe operator (`|`) to compose these steps into a graph. Each component in the pipe implements the `Runnable` interface, which means it supports `.invoke()`, `.stream()`, and `.batch()` consistently. This uniformity is what makes orchestration composable: you can swap any step for another without rewriting the surrounding pipeline.

The key insight with workflow orchestration is that you are separating concerns. The prompt template is responsible for formatting. The LLM is responsible for reasoning. The output parser is responsible for extracting structure. The orchestration layer is responsible for sequencing. When each concern is isolated, the system is easier to test, debug, and modify.

There are three fundamental orchestration patterns you will encounter in enterprise systems. Sequential orchestration runs steps one after another, where each step has access to all outputs from previous steps. Parallel orchestration runs independent steps simultaneously, reducing total wall-clock time. Conditional orchestration inspects intermediate outputs and routes to different sub-pipelines depending on what it finds. All three are demonstrated below.

In [4]:
# Pattern 1A: Sequential Orchestration
#
# This pattern is appropriate when each step depends on the output of the previous step.
# The pipeline below processes a procurement request through three stages:
# extraction, risk assessment, and decision recommendation.
# Each stage receives the outputs of all prior stages via RunnablePassthrough.assign.

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = get_llm(temperature=0.1)
str_out = StrOutputParser()

# Step 1: Extract structured information from the raw document
extract_chain = (
    ChatPromptTemplate.from_template(
        "Extract the following fields from this procurement request as a plain list.\n"
        "Fields: vendor name, contract value, duration, deliverable, payment terms.\n\n"
        "Request:\n{document}"
    ) | llm | str_out
)

# Step 2: Assess risk using the extracted fields
# Note that this prompt receives {extracted} which is the output of step 1
risk_chain = (
    ChatPromptTemplate.from_template(
        "Given these procurement details:\n{extracted}\n\n"
        "Identify the top 3 risks in this contract. Be specific about financial, "
        "delivery, and compliance risks. Rate each risk: Low, Medium, or High."
    ) | llm | str_out
)

# Step 3: Recommend a decision using both extracted data and risk assessment
decision_chain = (
    ChatPromptTemplate.from_template(
        "Procurement details:\n{extracted}\n\n"
        "Risk assessment:\n{risks}\n\n"
        "Based on the above, provide a procurement recommendation: Approve, Approve with conditions, "
        "or Escalate. State your reasoning in 3 sentences."
    ) | llm | str_out
)

# Assemble the sequential pipeline.
# RunnablePassthrough.assign adds new keys to the running state dict
# without overwriting existing ones. This means every downstream step
# can see everything that was computed upstream.
sequential_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(extracted=extract_chain)
    | RunnablePassthrough.assign(risks=risk_chain)
    | RunnablePassthrough.assign(decision=decision_chain)
)

sample_procurement = """
We are requesting approval for a 24-month software licensing agreement with CloudVendor Pvt Ltd
for their data observability platform. Total contract value: USD 420,000 payable quarterly.
Deliverable: platform access for up to 500 users with 99.5% SLA. Vendor is a 3-year-old startup
headquartered in Singapore with no existing relationship with our organisation.
Payment terms: 30% upfront, 70% spread across remaining quarters.
"""

result = sequential_pipeline.invoke(sample_procurement)

print("EXTRACTED FIELDS")
print("-" * 50)
print(result["extracted"])
print("\nRISK ASSESSMENT")
print("-" * 50)
print(result["risks"])
print("\nDECISION RECOMMENDATION")
print("-" * 50)
print(result["decision"])

EXTRACTED FIELDS
--------------------------------------------------
- Vendor name: CloudVendor Pvt Ltd  
- Contract value: USD 420,000  
- Duration: 24 months  
- Deliverable: Platform access for up to 500 users with 99.5% SLA  
- Payment terms: 30% upfront, 70% spread across remaining quarters  

RISK ASSESSMENT
--------------------------------------------------
Here are the top 3 risks identified in the contract, categorized into financial, delivery, and compliance risks, along with their ratings:

---

### **1. Financial Risk: Payment Structure**
**Risk:** The payment terms require 30% upfront payment, which exposes the organization to financial risk if the vendor fails to deliver as promised or goes out of business early in the contract. Additionally, spreading the remaining 70% across quarters may complicate cash flow management if the vendor's performance is inconsistent.  
**Rating:** **Medium**  
**Reason:** While upfront payments are common, they increase exposure to vendor-re

In [5]:
# Pattern 1B: Parallel Orchestration
#
# Use this when multiple independent analyses need to run on the same input.
# RunnableParallel dispatches all branches concurrently and collects results
# into a single dict. Wall-clock time equals the slowest branch, not the sum.

from langchain_core.runnables import RunnableParallel
import time

llm = get_llm(temperature=0.2)

# Three independent analysis dimensions on the same document
legal_analysis = (
    ChatPromptTemplate.from_template(
        "From a legal perspective, what are the key contractual obligations "
        "and liabilities in this document? Be concise.\n\n{document}"
    ) | llm | str_out
)

financial_analysis = (
    ChatPromptTemplate.from_template(
        "From a financial perspective, assess the payment structure, total cost of ownership, "
        "and any hidden costs in this document. Be concise.\n\n{document}"
    ) | llm | str_out
)

operational_analysis = (
    ChatPromptTemplate.from_template(
        "From an operations perspective, what delivery risks, dependency risks, "
        "and integration challenges exist in this document? Be concise.\n\n{document}"
    ) | llm | str_out
)

parallel_review = RunnableParallel(
    legal=legal_analysis,
    financial=financial_analysis,
    operational=operational_analysis,
)

start = time.time()
reviews = parallel_review.invoke({"document": sample_procurement})
elapsed = time.time() - start

print(f"All 3 analyses completed in {elapsed:.1f}s (parallel execution)")
print("\nLEGAL ANALYSIS")
print("-" * 50)
print(reviews["legal"])
print("\nFINANCIAL ANALYSIS")
print("-" * 50)
print(reviews["financial"])
print("\nOPERATIONAL ANALYSIS")
print("-" * 50)
print(reviews["operational"])

All 3 analyses completed in 7.8s (parallel execution)

LEGAL ANALYSIS
--------------------------------------------------
Key contractual obligations and liabilities:

1. **Deliverables**: CloudVendor Pvt Ltd must provide platform access for up to 500 users with a 99.5% SLA (Service Level Agreement). Failure to meet the SLA may result in liability for service deficiencies.

2. **Payment Terms**: The client must pay 30% upfront and the remaining 70% in quarterly installments over 24 months. Late payments may incur penalties or interest.

3. **Contract Value**: Total payment obligation is USD 420,000.

4. **Duration**: The agreement spans 24 months, binding both parties for this period.

5. **Vendor Risk**: As a 3-year-old startup with no prior relationship, there may be risks related to vendor reliability or financial stability.



FINANCIAL ANALYSIS
--------------------------------------------------
**Assessment:**

1. **Payment Structure**:  
   - 30% upfront ($126,000) and 70% ($294,0

In [6]:
# Pattern 1C: Batch Orchestration
#
# When you need to apply the same pipeline to a collection of inputs,
# use .batch() rather than looping with .invoke(). LangChain dispatches
# batch calls concurrently up to the configured concurrency limit.
# This is the standard pattern for bulk document processing, ticket triage,
# and any scenario where the same operation applies to many records.

triage_chain = (
    ChatPromptTemplate.from_template(
        "Classify this support ticket into one category: billing / technical / access / other.\n"
        "Also assign priority: P1 (critical) / P2 (high) / P3 (normal) / P4 (low).\n"
        "Respond as: Category: X | Priority: Y\n\nTicket: {ticket}"
    ) | llm | str_out
)

tickets = [
    {"ticket": "Production API returning 500 errors for all customers since 08:00 UTC. Revenue impact ongoing."},
    {"ticket": "My invoice shows a charge I do not recognise from last month."},
    {"ticket": "New team member cannot log in after account was provisioned yesterday."},
    {"ticket": "The export to CSV button on the reports page is missing a column."},
    {"ticket": "We need to discuss our enterprise contract renewal due next quarter."},
]

results = triage_chain.batch(tickets)

print("BATCH TRIAGE RESULTS")
print("-" * 60)
for ticket, classification in zip(tickets, results):
    print(f"Ticket : {ticket['ticket'][:70]}...")
    print(f"Result : {classification}")
    print()

BATCH TRIAGE RESULTS
------------------------------------------------------------
Ticket : Production API returning 500 errors for all customers since 08:00 UTC....
Result : Category: Technical | Priority: P1

Ticket : My invoice shows a charge I do not recognise from last month....
Result : Category: Billing | Priority: P2

Ticket : New team member cannot log in after account was provisioned yesterday....
Result : Category: Access | Priority: P2

Ticket : The export to CSV button on the reports page is missing a column....
Result : Category: Technical | Priority: P2

Ticket : We need to discuss our enterprise contract renewal due next quarter....
Result : Category: Billing | Priority: P3



**When to use each pattern**

Sequential orchestration is the right choice when each step meaningfully depends on previous outputs. Risk assessment cannot happen before extraction; decision cannot happen before risk assessment. The dependency chain is real and the sequential pattern makes it explicit.

Parallel orchestration is the right choice when analyses are independent. Legal, financial, and operational review of the same document do not depend on each other. Running them in parallel halves or thirds your latency without any loss of quality.

Batch orchestration is the right choice when you have many homogeneous inputs and want to process them all efficiently. Use it for overnight processing jobs, bulk classification, and any scenario where the same prompt applies to a collection of records.

## Section 2: Intermediate Reasoning

### Concept

Intermediate reasoning refers to the practice of making the model think step-by-step before producing a final answer. Without explicit reasoning instructions, LLMs often jump to conclusions. This works for simple factual queries but fails on complex problems where the answer depends on correctly working through intermediate sub-problems.

There are three reasoning strategies covered in this section. Chain-of-Thought prompting instructs the model to reason through a problem explicitly before answering. This is a prompting technique: you do not change the chain structure, only the prompt. ReAct (Reasoning and Acting) is a pattern where the model alternates between reasoning steps and tool calls, using observations from tools to refine its reasoning. Self-reflection is a pipeline pattern where a second LLM call critiques the first output and produces an improved version.

The practical difference between these strategies is significant. Chain-of-Thought works well for problems that require logical deduction but no external information. ReAct works well when the problem requires looking up information, calling APIs, or executing code. Self-reflection works well when output quality is critical and you can afford the latency of a second LLM call to improve the first.

In enterprise systems, intermediate reasoning is most valuable in high-stakes workflows: contract analysis, compliance checking, architectural review, financial modelling. These are precisely the workflows where a wrong answer has real consequences and where the model's reasoning process, not just its conclusion, matters for auditability.

In [7]:
# Reasoning Strategy 1: Chain-of-Thought (CoT)
#
# The key change versus a standard prompt is the instruction to reason step by step
# and show that reasoning before giving a final answer. This forces the model to
# decompose the problem rather than pattern-matching to a surface-level answer.
#
# You will see two prompts below: one without CoT and one with CoT on the same
# problem. Compare the quality and reliability of the outputs.

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0.0)
str_out = StrOutputParser()

problem = """
Our data team runs a nightly ETL job that processes 50GB of transaction data.
The job currently takes 4 hours on a single Azure VM (Standard_D8s_v3, 8 vCPUs, 32GB RAM).
We need to reduce this to under 1 hour. The job is embarrassingly parallel across
10 customer segments. Storage is Azure Blob. Budget is fixed at current VM cost.
What is the recommended architecture change?
"""

# Without Chain-of-Thought
standard_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior Azure architect."),
    ("human",  "{problem}")
])
standard_chain = standard_prompt | llm | str_out

# With Chain-of-Thought
# The system prompt instructs explicit step-by-step reasoning.
# The format section at the end structures the output for readability.
cot_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior Azure architect. When solving architecture problems:\n"
     "1. First restate the constraints clearly.\n"
     "2. Identify the bottleneck or root cause.\n"
     "3. Consider at least two solution approaches.\n"
     "4. Evaluate each approach against the constraints.\n"
     "5. State your final recommendation with justification.\n"
     "Show all of these steps explicitly before giving your recommendation."),
    ("human", "{problem}")
])
cot_chain = cot_prompt | llm | str_out

print("WITHOUT CHAIN-OF-THOUGHT")
print("=" * 60)
standard_answer = standard_chain.invoke({"problem": problem})
print(standard_answer)

print("\nWITH CHAIN-OF-THOUGHT")
print("=" * 60)
cot_answer = cot_chain.invoke({"problem": problem})
print(cot_answer)

WITHOUT CHAIN-OF-THOUGHT
To reduce the ETL job runtime to under 1 hour while keeping the budget fixed at the current VM cost, you can leverage **horizontal scaling** by splitting the workload across multiple smaller VMs. Since the job is embarrassingly parallel across 10 customer segments, you can process each segment independently on separate VMs. Here's the recommended architecture change:

### Recommended Architecture:
1. **Use Azure Virtual Machine Scale Sets (VMSS):**
   - Replace the single `Standard_D8s_v3` VM with a **scale set** of smaller VMs, such as `Standard_D2s_v3` (2 vCPUs, 8GB RAM).
   - The cost of 4 `Standard_D2s_v3` VMs is approximately equal to the cost of 1 `Standard_D8s_v3` VM, allowing you to stay within budget.

2. **Parallelize the ETL Job:**
   - Divide the ETL job into 10 independent tasks, one for each customer segment.
   - Assign each task to a separate VM in the scale set.
   - Use a **job orchestration tool** like Azure Batch, Azure Logic Apps, or Azure 

In [8]:
# Reasoning Strategy 2: ReAct Pattern (Reasoning + Acting)
#
# ReAct interleaves reasoning steps with tool calls. The model reasons about
# what it needs to know, calls a tool to find out, observes the result, and
# continues reasoning. This loop repeats until the model has enough information
# to produce a final answer.
#
# The pattern is: Thought -> Action -> Observation -> Thought -> ... -> Final Answer
#
# In LangChain v1.0+, create_agent from langchain.agents implements ReAct
# natively. The tools below simulate Azure resource lookups.

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.messages import AIMessage
import json, random

llm = get_llm(temperature=0.0)

@tool
def get_vm_metrics(vm_name: str) -> str:
    """Retrieve current CPU, memory, and disk utilisation for an Azure VM.
    Use this when you need to assess whether a VM is over or under-provisioned."""
    return json.dumps({
        "vm_name": vm_name,
        "cpu_utilisation_avg_7d": f"{random.randint(5, 92)}%",
        "memory_utilisation_avg_7d": f"{random.randint(20, 88)}%",
        "disk_iops_avg_7d": random.randint(100, 8000),
        "vm_size": "Standard_D8s_v3",
        "monthly_cost_usd": 486.40
    })

@tool
def get_alternative_vm_sizes(workload_type: str) -> str:
    """Retrieve recommended alternative Azure VM sizes for a given workload type.
    Workload types: compute-intensive, memory-intensive, balanced, burstable.
    Use this to compare sizing options."""
    options = {
        "compute-intensive": [
            {"size": "Standard_F8s_v2", "vcpus": 8,  "memory_gb": 16, "monthly_usd": 338},
            {"size": "Standard_F16s_v2","vcpus": 16, "memory_gb": 32, "monthly_usd": 676},
        ],
        "memory-intensive": [
            {"size": "Standard_E8s_v5", "vcpus": 8,  "memory_gb": 64, "monthly_usd": 504},
            {"size": "Standard_E16s_v5","vcpus": 16, "memory_gb": 128,"monthly_usd": 1008},
        ],
        "balanced": [
            {"size": "Standard_D4s_v5", "vcpus": 4,  "memory_gb": 16, "monthly_usd": 192},
            {"size": "Standard_D8s_v5", "vcpus": 8,  "memory_gb": 32, "monthly_usd": 384},
        ],
        "burstable": [
            {"size": "Standard_B4ms",   "vcpus": 4,  "memory_gb": 16, "monthly_usd": 152},
            {"size": "Standard_B8ms",   "vcpus": 8,  "memory_gb": 32, "monthly_usd": 304},
        ],
    }
    return json.dumps(options.get(workload_type, options["balanced"]))

@tool
def calculate_annual_savings(current_monthly_usd: float, proposed_monthly_usd: float) -> str:
    """Calculate annual savings and payback period when switching VM sizes.
    Use this after identifying a cheaper alternative to quantify the business case."""
    monthly_saving = current_monthly_usd - proposed_monthly_usd
    annual_saving  = monthly_saving * 12
    return json.dumps({
        "monthly_saving_usd": round(monthly_saving, 2),
        "annual_saving_usd":  round(annual_saving, 2),
        "saving_percentage":  f"{(monthly_saving / current_monthly_usd * 100):.1f}%"
    })

tools = [get_vm_metrics, get_alternative_vm_sizes, calculate_annual_savings]

# create_agent from langchain.agents is the v1.0+ API
# It builds a graph-based ReAct agent using LangGraph internally
react_agent = create_agent(
    llm,
    tools=tools,
    name="vm_optimisation_agent",
    system_prompt=(
        "You are an Azure FinOps specialist. Reason through problems step by step. "
        "Always gather metrics before making recommendations. "
        "Always quantify savings before concluding. "
        "Never answer without calling tools to gather evidence first."
    ),
)

result = react_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Analyse the VM named etl-prod-vm-01 and recommend whether we should right-size it. "
            "If a cheaper alternative exists, calculate the annual savings."
        )
    }]
})

print("REACT AGENT FINAL ANSWER")
print("=" * 60)
print(result["messages"][-1].content)

REACT AGENT FINAL ANSWER
The current VM, **etl-prod-vm-01**, is underutilized with an average CPU utilization of 11% and memory utilization of 61%. It is currently using the **Standard_D8s_v3** size, costing $486.40 per month.

A more cost-effective alternative is the **Standard_D4s_v5** size, which offers 4 vCPUs and 16 GB of memory at $192 per month. This size aligns better with the observed workload.

Switching to the **Standard_D4s_v5** would result in:
- **Monthly savings**: $294.40
- **Annual savings**: $3,532.80
- **Savings percentage**: 60.5%

I recommend right-sizing the VM to **Standard_D4s_v5** to optimize costs while meeting workload requirements.


In [9]:
# Reasoning Strategy 3: Self-Reflection Loop
#
# Self-reflection is a pipeline pattern where a second LLM call reviews
# the first output and either approves it or produces an improved version.
#
# This is particularly useful when output quality directly affects downstream
# decisions. A first-pass answer that is plausible but flawed can be caught
# and corrected before it leaves the pipeline.
#
# The pattern has three stages:
#   Stage 1: Generate an initial answer (temperature slightly elevated for creativity)
#   Stage 2: Critique the answer (temperature=0 for consistent evaluation)
#   Stage 3: Revise the answer based on the critique

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

generator_llm = get_llm(temperature=0.4, max_tokens=600)
critic_llm    = get_llm(temperature=0.0, max_tokens=400)
reviser_llm   = get_llm(temperature=0.2, max_tokens=600)
str_out = StrOutputParser()

# Stage 1: Generate initial response
generate_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a senior cloud architect. Provide a thorough initial answer."),
        ("human",  "{question}")
    ]) | generator_llm | str_out
)

# Stage 2: Critique the initial response
# The critic uses a structured rubric to evaluate the answer.
# It does not just say whether the answer is good; it identifies specific gaps.
critique_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a rigorous technical reviewer. Evaluate the answer against these criteria:\n"
         "1. Technical accuracy: are all claims correct?\n"
         "2. Completeness: are important considerations missing?\n"
         "3. Specificity: are recommendations concrete or vague?\n"
         "4. Actionability: can a senior engineer act on this immediately?\n\n"
         "List specific improvements needed. If no improvements are needed, say: APPROVED."),
        ("human",
         "Original question: {question}\n\nAnswer to review:\n{initial_answer}")
    ]) | critic_llm | str_out
)

# Stage 3: Revise the answer incorporating the critique
revise_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a senior cloud architect. Revise the answer below to address "
         "all critique points. If the critique says APPROVED, return the original answer unchanged."),
        ("human",
         "Original question: {question}\n\n"
         "Initial answer:\n{initial_answer}\n\n"
         "Critique:\n{critique}\n\n"
         "Revised answer:")
    ]) | reviser_llm | str_out
)

# Compose the self-reflection pipeline
self_reflection_pipeline = (
    {"question": RunnablePassthrough()}
    | RunnablePassthrough.assign(initial_answer=generate_chain)
    | RunnablePassthrough.assign(critique=critique_chain)
    | RunnablePassthrough.assign(final_answer=revise_chain)
)

question = (
    "Our Azure Kubernetes Service cluster is experiencing intermittent pod evictions "
    "during peak load. Node pool has 5 nodes of Standard_D4s_v3. "
    "What should we investigate and how should we resolve this?"
)

output = self_reflection_pipeline.invoke(question)

print("INITIAL ANSWER")
print("-" * 60)
print(output["initial_answer"])
print("\nCRITIQUE")
print("-" * 60)
print(output["critique"])
print("\nFINAL REVISED ANSWER")
print("-" * 60)
print(output["final_answer"])

INITIAL ANSWER
------------------------------------------------------------
Intermittent pod evictions during peak load in your Azure Kubernetes Service (AKS) cluster are typically indicative of resource constraints or misconfigurations. The `Standard_D4s_v3` nodes in your node pool provide 4 vCPUs and 16 GB of RAM per node, but during peak load, resource contention can lead to pod evictions. Below is a thorough investigation and resolution plan:

---

### **1. Investigate Resource Usage**
#### **a. Node Resource Utilization**
- Check CPU and memory utilization on each node during peak load using:
  ```
  kubectl top nodes
  ```
- If nodes are running consistently near their limits, pods may be evicted due to insufficient resources.

#### **b. Pod Resource Requests and Limits**
- Inspect the resource requests and limits defined for your pods:
  ```
  kubectl describe pod <pod-name>
  ```
- If requests are too high or limits are not defined, Kubernetes may struggle to schedule pods effi

## Section 3: Validation Prompts

### Concept

LLM outputs cannot be unconditionally trusted. A model can produce a well-structured, confident-sounding response that contains factual errors, omits critical constraints, or fails to meet the format requirements of a downstream system. In production orchestration pipelines, unvalidated LLM outputs that flow into databases, APIs, or decision systems create silent failures that are difficult to trace.

Validation prompts address this by inserting a dedicated verification step between the LLM output and the next stage of the pipeline. The validator is a second LLM call (or a structured Pydantic check, or both) that asks: does this output meet the criteria it needs to meet?

There are two layers of validation worth distinguishing. Structural validation checks whether the output is in the expected format: is the JSON parseable, are all required fields present, are numeric values within expected ranges? This can often be done with Pydantic without an LLM call. Semantic validation checks whether the output is correct and appropriate: does the recommendation contradict the constraints, does the summary omit a critical fact, does the classification match the content? This requires an LLM call because it involves understanding meaning.

In enterprise pipelines, validation is especially important at boundary points: before writing to a database, before sending an email, before routing a document to a human reviewer. The cost of a validation LLM call is small relative to the cost of a silent error propagating through the system.

In [10]:
# Validation Layer 1: Structural Validation with Pydantic
#
# Use Pydantic to enforce that LLM outputs conform to a schema before
# they are used downstream. If the LLM returns malformed JSON or omits
# a required field, the validation catches it immediately.
#
# with_structured_output() is the modern LangChain API for this.
# It handles schema injection into the prompt and parsing automatically.

from pydantic import BaseModel, Field, field_validator
from typing import Literal, List
from langchain_core.prompts import ChatPromptTemplate

llm = get_llm(temperature=0.0)

# Define the expected output schema with validation rules
class IncidentAssessment(BaseModel):
    severity:          Literal["P1", "P2", "P3", "P4"] = Field(description="Incident severity level")
    affected_systems:  List[str]                         = Field(description="List of affected systems or services")
    estimated_impact:  str                               = Field(description="Business impact description in one sentence")
    immediate_actions: List[str]                         = Field(description="List of immediate actions to take, maximum 5")
    escalate_to:       str                               = Field(description="Role or team to escalate to")
    confidence_score:  float                             = Field(description="Confidence in this assessment, 0.0 to 1.0")

    @field_validator("confidence_score")
    @classmethod
    def validate_confidence(cls, v):
        if not 0.0 <= v <= 1.0:
            raise ValueError("confidence_score must be between 0.0 and 1.0")
        return v

    @field_validator("immediate_actions")
    @classmethod
    def validate_actions_count(cls, v):
        if len(v) > 5:
            raise ValueError("immediate_actions must contain 5 items or fewer")
        return v

# with_structured_output enforces the Pydantic schema on the LLM output
structured_llm = llm.with_structured_output(IncidentAssessment)

assessment_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an on-call incident commander. Assess the incident and produce a "
     "structured assessment. Be precise and conservative: if unsure of severity, go higher."),
    ("human", "{incident_report}")
])

structured_assessment_chain = assessment_prompt | structured_llm

incident = """
ALERT: Azure SQL Database in East US region reporting connection pool exhaustion.
Error rate on checkout service: 34% over the last 15 minutes.
Customer complaints coming in via support channel. Conversion rate has dropped.
Database DTU usage at 98%. Root cause unknown. On-call engineer just paged.
"""

assessment: IncidentAssessment = structured_assessment_chain.invoke({"incident_report": incident})

print("STRUCTURED INCIDENT ASSESSMENT")
print("=" * 60)
print(f"Severity         : {assessment.severity}")
print(f"Affected Systems : {', '.join(assessment.affected_systems)}")
print(f"Impact           : {assessment.estimated_impact}")
print(f"Escalate To      : {assessment.escalate_to}")
print(f"Confidence       : {assessment.confidence_score:.0%}")
print(f"Immediate Actions:")
for i, action in enumerate(assessment.immediate_actions, 1):
    print(f"  {i}. {action}")

STRUCTURED INCIDENT ASSESSMENT
Severity         : P1
Affected Systems : Azure SQL Database (East US), Checkout Service
Impact           : High error rates in checkout service leading to customer complaints and reduced conversion rates.
Escalate To      : Database Operations Team
Confidence       : 90%
Immediate Actions:
  1. Notify the database team to investigate and mitigate the DTU usage issue.
  2. Scale up the Azure SQL Database to alleviate immediate resource constraints.
  3. Implement connection pool limits or retries in the checkout service to reduce error rates.
  4. Communicate with customer support to inform affected users of the issue.
  5. Monitor the system for further degradation or improvement.


c:\Python\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=IncidentAssessment(severi...', confidence_score=0.9), input_type=IncidentAssessment])
  return self.__pydantic_serializer__.to_python(


In [11]:
# Validation Layer 2: Semantic Validation Prompt
#
# Pydantic checks structure but not meaning. A semantically wrong answer
# can pass structural validation perfectly. A second LLM call evaluates
# the output against criteria that require understanding the content.
#
# The validator returns a structured verdict: pass/fail plus a reason.
# The pipeline can then retry, escalate, or pass through based on this verdict.

from pydantic import BaseModel, Field
from typing import Literal, List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

llm          = get_llm(temperature=0.3, max_tokens=800)
validator_llm = get_llm(temperature=0.0, max_tokens=300)
str_out      = StrOutputParser()

class ValidationVerdict(BaseModel):
    verdict:        Literal["PASS", "FAIL"]  = Field(description="Whether the output passes validation")
    issues:         List[str]                = Field(description="List of specific issues found. Empty if PASS.")
    recommendation: str                      = Field(description="What should happen next: proceed, retry, or escalate")

structured_validator = validator_llm.with_structured_output(ValidationVerdict)

# The generator produces a compliance recommendation
generate_recommendation = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a data governance advisor. Provide a compliance recommendation."),
        ("human",  "{scenario}")
    ]) | llm | str_out
)

# The validator checks whether the recommendation meets specific criteria
validate_recommendation = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are a compliance officer reviewing AI-generated recommendations. "
         "A recommendation PASSES if it meets ALL of these criteria:\n"
         "1. References at least one specific regulation or standard (e.g. GDPR, ISO 27001, HIPAA).\n"
         "2. Contains at least one concrete, actionable step (not just general advice).\n"
         "3. Does not contradict the constraints stated in the original scenario.\n"
         "4. Does not recommend storing regulated data in a non-compliant region.\n"
         "A recommendation FAILS if it violates any of these criteria."),
        ("human",
         "Original scenario: {scenario}\n\nRecommendation to validate:\n{recommendation}")
    ]) | structured_validator
)

# Pipeline that generates, validates, and reports
validated_pipeline = (
    {"scenario": RunnablePassthrough()}
    | RunnablePassthrough.assign(recommendation=generate_recommendation)
    | RunnablePassthrough.assign(validation=validate_recommendation)
)

scenario = """
Our HR team wants to store employee performance review data in Azure Blob Storage
in the South East Asia region. Employees are based in Germany and France.
Data includes names, manager ratings, salary bands, and disciplinary records.
We need guidance on whether this is permitted and what controls are required.
"""

output = validated_pipeline.invoke(scenario)
verdict: ValidationVerdict = output["validation"]

print("GENERATED RECOMMENDATION")
print("-" * 60)
print(output["recommendation"])
print("\nVALIDATION VERDICT")
print("-" * 60)
print(f"Verdict        : {verdict.verdict}")
print(f"Recommendation : {verdict.recommendation}")
if verdict.issues:
    print(f"Issues found   :")
    for issue in verdict.issues:
        print(f"  - {issue}")

GENERATED RECOMMENDATION
------------------------------------------------------------
Storing employee performance review data in Azure Blob Storage located in the South East Asia region raises compliance concerns due to the stringent data protection laws in Germany and France, which are governed by the EU General Data Protection Regulation (GDPR). Here are my recommendations:

### 1. **Evaluate Data Transfer Compliance**
   - **GDPR Cross-Border Data Transfer Rules**: Personal data of employees based in Germany and France cannot be transferred or stored outside the European Economic Area (EEA) unless certain safeguards are in place. South East Asia is not within the EEA, so you must ensure compliance with GDPR's data transfer requirements.
   - **Adequacy Decision**: Check if the South East Asia region has an adequacy decision from the European Commission. If not, you will need additional safeguards.
   - **Standard Contractual Clauses (SCCs)**: Implement SCCs between your organizatio

c:\Python\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ValidationVerdict(verdict...ecommendation='proceed'), input_type=ValidationVerdict])
  return self.__pydantic_serializer__.to_python(


In [12]:
# Validation Layer 3: Auto-Retry on Validation Failure
#
# A validation check is only useful if the pipeline can act on a failure.
# This pattern retries the generation step with an enriched prompt that
# includes the validation feedback, giving the model an opportunity to
# correct the specific issues that caused the failure.
#
# The retry is bounded (max_retries) to prevent infinite loops.

def run_with_validation_retry(scenario: str, max_retries: int = 2) -> dict:
    """
    Run the validated pipeline with automatic retry on validation failure.
    On failure, the generation prompt is enriched with the specific issues
    identified by the validator so the model can correct them.
    """
    retry_llm = get_llm(temperature=0.2, max_tokens=800)
    attempt = 0
    prior_issues = []

    while attempt <= max_retries:
        # On retry, enrich the prompt with previous validation feedback
        if prior_issues:
            enriched_scenario = (
                f"{scenario}\n\n"
                f"Previous attempt had these issues that must be addressed:\n"
                + "\n".join(f"- {i}" for i in prior_issues)
            )
        else:
            enriched_scenario = scenario

        enriched_generate = (
            ChatPromptTemplate.from_messages([
                ("system", "You are a data governance advisor. Provide a compliance recommendation."),
                ("human",  "{scenario}")
            ]) | retry_llm | StrOutputParser()
        )

        recommendation = enriched_generate.invoke({"scenario": enriched_scenario})

        validation_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are a compliance officer. A recommendation PASSES if it: "
             "(1) references a specific regulation, (2) contains concrete actions, "
             "(3) does not contradict the scenario constraints."),
            ("human", "Scenario: {scenario}\n\nRecommendation: {recommendation}")
        ]) | validator_llm.with_structured_output(ValidationVerdict)

        verdict: ValidationVerdict = validation_prompt.invoke({
            "scenario": scenario,
            "recommendation": recommendation
        })

        print(f"Attempt {attempt + 1}: {verdict.verdict}")

        if verdict.verdict == "PASS":
            return {"recommendation": recommendation, "verdict": verdict, "attempts": attempt + 1}

        prior_issues = verdict.issues
        attempt += 1

    # If all retries exhausted, return last attempt with failure noted
    return {
        "recommendation": recommendation,
        "verdict": verdict,
        "attempts": attempt,
        "status": "ESCALATE: validation failed after maximum retries"
    }

print("RUNNING VALIDATED PIPELINE WITH AUTO-RETRY")
print("=" * 60)
final = run_with_validation_retry(scenario)
print(f"\nResolved in {final['attempts']} attempt(s)")
print(f"Final verdict: {final['verdict'].verdict}")
print(f"\nFinal recommendation:")
print(final["recommendation"])

RUNNING VALIDATED PIPELINE WITH AUTO-RETRY
Attempt 1: PASS

Resolved in 1 attempt(s)
Final verdict: PASS

Final recommendation:
Storing employee performance review data in Azure Blob Storage in the South East Asia region raises significant compliance considerations due to data protection laws, particularly the General Data Protection Regulation (GDPR), which applies to employees based in Germany and France. Here are my recommendations:

### 1. **Assess GDPR Compliance**
   - **Data Transfer Restrictions**: GDPR imposes strict rules on transferring personal data outside the European Economic Area (EEA). Storing data in South East Asia constitutes a data transfer to a non-EEA region. You must ensure that the transfer complies with GDPR requirements.
   - **Adequacy Decision**: Check if the South East Asia region has an adequacy decision from the European Commission. If not, you must implement additional safeguards.
   - **Appropriate Safeguards**: If no adequacy decision exists, use mech

c:\Python\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ValidationVerdict(verdict...ides actionable steps.'), input_type=ValidationVerdict])
  return self.__pydantic_serializer__.to_python(


## Section 4: Branching

### Concept

Branching is the mechanism by which an orchestration pipeline routes inputs to different sub-pipelines based on their content, classification, or some computed property. Without branching, a pipeline can only handle inputs that are uniform enough for a single prompt to address well. In practice, enterprise inputs are heterogeneous: a support queue contains billing questions, technical bugs, feature requests, and escalations, each of which deserves a different handling path.

Branching in LangChain is implemented with `RunnableBranch`. It takes a list of condition-handler pairs. Each condition is a Python callable that receives the current state dict and returns a boolean. The first condition that evaluates to True determines which handler chain receives the input. If no condition matches, a default handler runs.

The branching condition can be based on anything available in the state: a prior classification step, a field extracted from the input, a user attribute injected at runtime. The most common pattern in enterprise systems is to run a lightweight classification chain first, then branch based on the classification result. This two-step pattern (classify then branch) keeps the routing logic clean and testable.

An important design consideration is that each branch can itself be a full sub-pipeline, not just a single prompt. A legal branch might include extraction, risk assessment, and escalation logic. A billing branch might include account lookup, charge validation, and resolution recommendation. Branching lets you compose specialised pipelines without those pipelines needing to know about each other.

In [13]:
# Branching Pattern 1: Classification-then-Branch
#
# Step 1: A lightweight classifier determines the document or query type.
# Step 2: RunnableBranch routes to the appropriate specialist chain.
#
# The classifier uses temperature=0 for consistent, deterministic routing.
# Specialist chains can use higher temperature where appropriate.

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

classifier_llm  = get_llm(temperature=0.0)
specialist_llm  = get_llm(temperature=0.3, max_tokens=600)
str_out = StrOutputParser()

# Classifier: returns one of four category labels
classifier = (
    ChatPromptTemplate.from_template(
        "Classify this document into exactly one category.\n"
        "Categories: contract / incident-report / architecture-proposal / general\n"
        "Respond with just the category label, lowercase, no punctuation.\n\n"
        "Document:\n{document}"
    ) | classifier_llm | str_out
)

# Specialist chains: each is tuned for its document type
contract_chain = (
    ChatPromptTemplate.from_template(
        "You are a contract analyst. Review this contract and extract:\n"
        "1. Parties involved\n2. Key obligations\n3. Termination conditions\n"
        "4. Liability caps\n5. Recommended action (sign / negotiate / reject)\n\n"
        "Contract:\n{document}"
    ) | specialist_llm | str_out
)

incident_chain = (
    ChatPromptTemplate.from_template(
        "You are an incident commander. For this incident report:\n"
        "1. Assign severity (P1-P4)\n2. Identify root cause hypothesis\n"
        "3. List immediate remediation steps\n4. Identify who must be notified\n\n"
        "Incident report:\n{document}"
    ) | specialist_llm | str_out
)

architecture_chain = (
    ChatPromptTemplate.from_template(
        "You are a principal architect. Review this architecture proposal against "
        "the Azure Well-Architected Framework. Cover: reliability, security, "
        "cost optimisation, operational excellence, and performance. "
        "Provide a verdict: Approve / Approve with conditions / Reject.\n\n"
        "Proposal:\n{document}"
    ) | specialist_llm | str_out
)

general_chain = (
    ChatPromptTemplate.from_template(
        "Provide a concise professional summary and any recommended actions "
        "for the following document:\n\n{document}"
    ) | specialist_llm | str_out
)

# Branch based on the classification result
branch = RunnableBranch(
    (lambda x: "contract"              in x["doc_type"].lower(), contract_chain),
    (lambda x: "incident"              in x["doc_type"].lower(), incident_chain),
    (lambda x: "architecture-proposal" in x["doc_type"].lower(), architecture_chain),
    general_chain,
)

# Full pipeline: classify, print route, then branch
branching_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(doc_type=classifier)
    | RunnablePassthrough.assign(
        _logged=lambda x: print(f"  Classified as: {x['doc_type'].strip()}") or ""
    )
    | RunnablePassthrough.assign(analysis=branch)
)

# Test with three different document types
test_docs = {
    "Contract": """
    SERVICE AGREEMENT between Contoso Ltd (Client) and Fabrikam Systems (Vendor).
    Vendor will deliver data migration services over 6 months for USD 180,000.
    Liability capped at total contract value. Either party may terminate with 30-day notice.
    Vendor guarantees 99% milestone delivery rate. Penalties apply for delays over 5 business days.
    """,
    "Incident Report": """
    INCIDENT-2024-0892: Azure Event Hub in West Europe region dropped 23% of messages
    between 14:00 and 15:30 UTC. Downstream analytics pipeline received incomplete data.
    Daily business intelligence reports will be inaccurate for this time window.
    Root cause under investigation. Throughput units were at maximum capacity.
    """,
    "Architecture Proposal": """
    PROPOSAL: Deploy customer-facing API on Azure Container Apps with Azure API Management
    as the gateway. Backend: Azure Cosmos DB with multi-region write enabled.
    Authentication: Azure AD B2C. Estimated traffic: 10,000 requests/minute peak.
    No disaster recovery plan documented. No WAF configured on the API Management instance.
    """,
}

for label, doc in test_docs.items():
    print(f"\n{'=' * 60}")
    print(f"Input type: {label}")
    print("=" * 60)
    result = branching_pipeline.invoke(doc)
    print(result["analysis"])


Input type: Contract
  Classified as: contract
Here is the analysis of the provided contract:

### 1. **Parties Involved**
   - **Client:** Contoso Ltd
   - **Vendor:** Fabrikam Systems

---

### 2. **Key Obligations**
   - **Vendor's Obligations:**
     - Deliver data migration services over a 6-month period.
     - Guarantee a 99% milestone delivery rate.
     - Accept penalties for delays exceeding 5 business days.
   - **Client's Obligations:**
     - Pay USD 180,000 for the services.

---

### 3. **Termination Conditions**
   - Either party may terminate the agreement with a 30-day notice.

---

### 4. **Liability Caps**
   - Liability is capped at the total contract value (USD 180,000).

---

### 5. **Recommended Action**
   - **Negotiate**:
     - The liability cap may be too restrictive for the Client, especially if the Vendor's failure causes significant financial or operational damage beyond USD 180,000.
     - Consider negotiating stricter penalties for milestone delivery d

In [14]:
# Branching Pattern 2: Multi-Level Branching
#
# Branches can themselves contain branches, enabling hierarchical routing.
# This is useful when a category has meaningful sub-categories that each
# warrant different handling.
#
# The example below adds a severity sub-branch inside the incident path:
# P1/P2 incidents get an escalation-focused response;
# P3/P4 incidents get a standard resolution path.

sev_classifier = (
    ChatPromptTemplate.from_template(
        "Classify this incident severity. P1: critical revenue impact. "
        "P2: significant degradation. P3: minor issue. P4: cosmetic/low priority.\n"
        "Reply with just: P1, P2, P3, or P4.\n\nIncident:\n{document}"
    ) | classifier_llm | str_out
)

p1p2_chain = (
    ChatPromptTemplate.from_template(
        "CRITICAL INCIDENT RESPONSE. This is a P1/P2 incident requiring immediate escalation.\n"
        "1. Who must be paged right now (roles)?\n"
        "2. What is the 15-minute action plan?\n"
        "3. What customer communication should go out within 30 minutes?\n\n"
        "Incident: {document}"
    ) | specialist_llm | str_out
)

p3p4_chain = (
    ChatPromptTemplate.from_template(
        "Standard incident resolution path.\n"
        "1. Assign to relevant team\n"
        "2. Provide resolution steps\n"
        "3. Estimated resolution time\n\n"
        "Incident: {document}"
    ) | specialist_llm | str_out
)

# Inner branch: severity-based routing within incident type
severity_branch = RunnableBranch(
    (lambda x: x["severity"].strip() in ["P1", "P2"], p1p2_chain),
    p3p4_chain,
)

# Multi-level incident pipeline
multilevel_incident_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(severity=sev_classifier)
    | RunnablePassthrough.assign(
        _logged=lambda x: print(f"  Severity assigned: {x['severity'].strip()}") or ""
    )
    | RunnablePassthrough.assign(response=severity_branch)
)

critical_incident = """
Payment processing service returning HTTP 500 for all transactions since 09:47 UTC.
Estimated lost revenue: USD 12,000 per minute. 47 customer complaints in last 10 minutes.
All payment methods affected. Root cause unknown. Database connections normal.
"""

minor_incident = """
Report export button on the analytics dashboard occasionally shows a spinner for
longer than expected before the file downloads. Affects approximately 2% of export attempts.
No data loss. Workaround: refresh and retry.
"""

for label, doc in [("Critical Incident", critical_incident), ("Minor Incident", minor_incident)]:
    print(f"\n{'=' * 60}")
    print(f"Input: {label}")
    print("=" * 60)
    r = multilevel_incident_pipeline.invoke(doc)
    print(r["response"])


Input: Critical Incident
  Severity assigned: P1
### CRITICAL INCIDENT RESPONSE PLAN

#### **1. Who must be paged right now (roles)?**
Immediate escalation requires notifying the following roles:
- **Incident Commander**: To lead the response and coordinate efforts.
- **Payment Service Engineering Team**: To investigate and resolve the issue.
- **Database Administrator (DBA)**: To confirm database health and assist in troubleshooting.
- **Site Reliability Engineer (SRE)**: To assess infrastructure and system health.
- **Customer Support Lead**: To manage customer communication and complaints.
- **Executive Stakeholder (e.g., CTO or VP of Engineering)**: To ensure leadership is informed and provide decision-making support.
- **Security Team**: To rule out potential security breaches or malicious activity.

#### **2. 15-Minute Action Plan**
**0–5 Minutes:**
- **Incident Commander**: Establish communication channels (e.g., Slack war room, Zoom call) and assign roles.
- **Engineering Team

## Section 5: Copilot Studio and No-Code Agentic Systems

### Concept

Microsoft Copilot Studio is a low-code platform for building conversational AI agents that can be deployed across Teams, websites, mobile apps, and third-party channels without writing backend code. It sits at the intersection of the pipelines you have built in this lab and the business users who need to interact with them.

Understanding Copilot Studio matters even for developers, because it defines the surface through which your LangChain pipelines will be consumed by non-technical users. The architecture of a production enterprise AI system typically has two layers: the orchestration layer (LangChain, Azure OpenAI, Python) which you control, and the conversation layer (Copilot Studio, Teams, web chat) which business users interact with. These two layers communicate through HTTP endpoints.

Copilot Studio agents are built around topics. A topic is a conversation flow triggered by a phrase or intent. Topics can call external actions (HTTP endpoints, Power Automate flows, Azure Functions) to retrieve information or execute operations. This is the integration point: your LangChain pipeline becomes an HTTP endpoint, and Copilot Studio calls it as an action within a topic.

No-code agentic systems in Copilot Studio use Generative AI orchestration mode, where the agent autonomously decides which topics and actions to invoke based on user intent. You define the actions (what can be done) and the knowledge sources (what can be retrieved), and the model decides how to use them. This mirrors the ReAct pattern from Section 2, but without writing any Python.

The cells below demonstrate how to expose a LangChain pipeline as an HTTP endpoint and how to structure the request/response format that Copilot Studio expects.

In [15]:
# Copilot Studio Integration: Exposing a LangChain Pipeline as an HTTP Endpoint
#
# Copilot Studio calls external actions via HTTP POST requests.
# The request body contains the user's input and any context variables.
# The response body must contain the fields Copilot Studio expects to
# display or use in subsequent conversation steps.
#
# This cell simulates the request/response cycle using the FastAPI pattern
# that you would deploy to Azure Container Apps or Azure Functions.
# The simulation runs inline without a running server.

from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Request schema: matches what Copilot Studio sends
class CopilotRequest(BaseModel):
    user_query:   str            = Field(description="The user's question or input")
    session_id:   str            = Field(description="Conversation session identifier")
    user_role:    Optional[str]  = Field(default="employee", description="User's role for context")
    department:   Optional[str]  = Field(default=None, description="User's department if known")

# Response schema: matches what Copilot Studio expects back
class CopilotResponse(BaseModel):
    answer:       str  = Field(description="The answer to display to the user")
    confidence:   str  = Field(description="High / Medium / Low")
    needs_human:  bool = Field(description="True if the query should be escalated to a human")
    follow_up:    str  = Field(description="Suggested follow-up question for the user")

llm = get_llm(temperature=0.2)
structured_llm = llm.with_structured_output(CopilotResponse)

# The pipeline that Copilot Studio will call
copilot_pipeline = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are an enterprise IT support assistant integrated into Microsoft Teams via Copilot Studio. "
         "The user's role is {user_role} in the {department} department. "
         "Answer IT support queries concisely. "
         "Set needs_human=True if the query requires account-level access, involves data loss, "
         "or is a P1/P2 incident. "
         "Confidence: High if you are certain, Medium if you have partial information, Low if you are guessing."),
        ("human", "{user_query}")
    ]) | structured_llm
)

# Simulate the HTTP handler function
# In production this would be a FastAPI route: @app.post("/copilot/it-support")
def handle_copilot_request(request: CopilotRequest) -> CopilotResponse:
    response: CopilotResponse = copilot_pipeline.invoke({
        "user_query":  request.user_query,
        "user_role":   request.user_role,
        "department":  request.department or "unknown",
    })
    return response

# Simulate three Copilot Studio requests
test_requests = [
    CopilotRequest(
        user_query="I cannot access SharePoint since this morning.",
        session_id="sess-001",
        user_role="manager",
        department="Finance"
    ),
    CopilotRequest(
        user_query="All users in our office lost VPN access 10 minutes ago.",
        session_id="sess-002",
        user_role="it-admin",
        department="IT"
    ),
    CopilotRequest(
        user_query="How do I set up my out-of-office reply in Outlook?",
        session_id="sess-003",
        user_role="employee",
        department="HR"
    ),
]

for req in test_requests:
    print(f"\nUSER ({req.user_role}, {req.department}): {req.user_query}")
    print("-" * 60)
    resp = handle_copilot_request(req)
    print(f"Answer        : {resp.answer}")
    print(f"Confidence    : {resp.confidence}")
    print(f"Needs Human   : {resp.needs_human}")
    print(f"Follow-up     : {resp.follow_up}")


USER (manager, Finance): I cannot access SharePoint since this morning.
------------------------------------------------------------


c:\Python\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CopilotResponse(answer='I...ent device or network?'), input_type=CopilotResponse])
  return self.__pydantic_serializer__.to_python(


Answer        : It seems you are experiencing access issues with SharePoint. Please ensure your internet connection is stable and try clearing your browser cache. If the issue persists, it may require account-level troubleshooting.
Confidence    : High
Needs Human   : True
Follow-up     : Have you tried accessing SharePoint from a different device or network?

USER (it-admin, IT): All users in our office lost VPN access 10 minutes ago.
------------------------------------------------------------


c:\Python\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CopilotResponse(answer='T...s and logs for errors?'), input_type=CopilotResponse])
  return self.__pydantic_serializer__.to_python(


Answer        : This issue might be related to a network or VPN server outage. Please check the VPN server status and logs for any errors.
Confidence    : High
Needs Human   : True
Follow-up     : Have you checked the VPN server status and logs for errors?

USER (employee, HR): How do I set up my out-of-office reply in Outlook?
------------------------------------------------------------
Answer        : To set up an out-of-office reply in Outlook, go to 'File' > 'Automatic Replies' (Out of Office), select 'Send automatic replies,' and configure the message and time range. If you use Outlook Web, go to 'Settings' > 'View all Outlook settings' > 'Mail' > 'Automatic replies.'
Confidence    : High
Needs Human   : False
Follow-up     : Do you need help with configuring the message or setting the time range?


c:\Python\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CopilotResponse(answer="T...etting the time range?'), input_type=CopilotResponse])
  return self.__pydantic_serializer__.to_python(


### How This Connects to Copilot Studio

Once you deploy `handle_copilot_request` as an Azure Function or Container App endpoint, you connect it to Copilot Studio by creating a Custom Connector in Power Platform. The connector describes your API schema (matching `CopilotRequest` and `CopilotResponse`) and Copilot Studio uses it to call your pipeline as an Action.

In the Copilot Studio canvas, you create a Topic triggered by phrases like "IT support", "I have a technical problem", or "something is broken". Inside that topic, you add an action node that calls your connector. The `answer` field from the response is displayed to the user. The `needs_human` field can trigger a transfer to a live agent using the built-in escalation node.

When you enable Generative AI orchestration on the agent, you no longer need to define explicit topics for every possible query. The model reads the action descriptions and decides when to call your pipeline. Your `system_prompt` in the pipeline and the action description in Copilot Studio together define the agent's behaviour.

## Section 6: Copilot Studio + Power Platform and Dataverse

### Concept

Power Platform is Microsoft's suite of low-code tools: Power Apps for building applications, Power Automate for workflow automation, Power BI for reporting, and Dataverse as the underlying data platform. Copilot Studio agents that need to read from or write to business data connect to Power Platform through these services.

Dataverse is the data layer that underpins all Power Platform applications. It stores structured business data in tables (formerly called entities), enforces relationships, and provides a security model based on roles and business units. When a Copilot Studio agent needs to create a support ticket, look up a customer record, or log an AI-generated recommendation, it writes to Dataverse.

From an architecture perspective, the integration works as follows. Your LangChain pipeline produces structured outputs (using Pydantic models as shown in Section 3). Those outputs are sent to an Azure Function or Power Automate HTTP trigger. Power Automate reads the payload and writes the relevant fields to Dataverse tables. Power Apps surfaces the Dataverse data to business users. Copilot Studio queries Dataverse via the built-in Dataverse connector to provide grounded, current answers.

The cells below demonstrate the Python side of this integration: calling the Dataverse Web API from Python to read and write records, and structuring LangChain outputs in a format that maps cleanly to Dataverse table schemas. You need a Power Platform environment with Dataverse enabled and an Azure AD app registration with Dataverse API permissions to run these cells against a real environment.

In [16]:
# Dataverse Integration: Writing LangChain Outputs to Dataverse
#
# Dataverse exposes a REST API (OData v4) at:
# https://YOUR_ORG.crm.dynamics.com/api/data/v9.2/
#
# Authentication uses Azure AD. You need:
#   - DATAVERSE_URL:   https://YOUR_ORG.crm.dynamics.com
#   - AZURE_TENANT_ID: your AAD tenant
#   - AZURE_CLIENT_ID: app registration client ID with Dataverse permissions
#   - AZURE_CLIENT_SECRET: app registration secret
#
# The DataverseClient class below wraps the common operations.
# The simulation mode (use_simulation=True) runs all logic locally
# without a real Dataverse connection, so you can test the pipeline
# structure before connecting to your environment.

import requests
import json
from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import datetime

DATAVERSE_URL     = os.environ.get("DATAVERSE_URL",        "https://YOUR_ORG.crm.dynamics.com")
AZURE_TENANT_ID   = os.environ.get("AZURE_TENANT_ID",      "YOUR_TENANT_ID")
AZURE_CLIENT_ID   = os.environ.get("AZURE_CLIENT_ID",      "YOUR_CLIENT_ID")
AZURE_CLIENT_SECRET = os.environ.get("AZURE_CLIENT_SECRET","YOUR_CLIENT_SECRET")

class DataverseClient:
    """
    Lightweight client for Dataverse Web API (OData v4).
    Handles token acquisition and CRUD operations.
    Set use_simulation=True to run without a real Dataverse environment.
    """

    def __init__(self, use_simulation: bool = True):
        self.use_simulation = use_simulation
        self.base_url = f"{DATAVERSE_URL}/api/data/v9.2"
        self._token = None

    def _get_token(self) -> str:
        """Acquire an OAuth2 token from Azure AD for Dataverse access."""
        token_url = f"https://login.microsoftonline.com/{AZURE_TENANT_ID}/oauth2/v2.0/token"
        resp = requests.post(token_url, data={
            "grant_type":    "client_credentials",
            "client_id":     AZURE_CLIENT_ID,
            "client_secret": AZURE_CLIENT_SECRET,
            "scope":         f"{DATAVERSE_URL}/.default",
        })
        resp.raise_for_status()
        return resp.json()["access_token"]

    def _headers(self) -> dict:
        if not self._token:
            self._token = self._get_token()
        return {
            "Authorization":    f"Bearer {self._token}",
            "Content-Type":     "application/json",
            "OData-MaxVersion": "4.0",
            "OData-Version":    "4.0",
            "Prefer":           "return=representation",
        }

    def create_record(self, table_name: str, record: dict) -> dict:
        """Create a record in a Dataverse table. Returns the created record with its ID."""
        if self.use_simulation:
            import uuid
            simulated = {**record, "cr_id": str(uuid.uuid4()), "createdon": datetime.utcnow().isoformat()}
            print(f"  [SIMULATION] Created record in {table_name}:")
            print(f"  {json.dumps(simulated, indent=2)}")
            return simulated

        resp = requests.post(
            f"{self.base_url}/{table_name}",
            headers=self._headers(),
            json=record
        )
        resp.raise_for_status()
        return resp.json()

    def query_records(self, table_name: str, filter_query: str, select_fields: List[str] = None) -> List[dict]:
        """Query Dataverse records with an OData filter expression."""
        if self.use_simulation:
            print(f"  [SIMULATION] Querying {table_name} where {filter_query}")
            return [{"cr_name": "Simulated Record", "cr_status": "open", "cr_priority": "high"}]

        params = {"$filter": filter_query}
        if select_fields:
            params["$select"] = ",".join(select_fields)

        resp = requests.get(
            f"{self.base_url}/{table_name}",
            headers=self._headers(),
            params=params
        )
        resp.raise_for_status()
        return resp.json().get("value", [])

# Initialise the client in simulation mode
# Set use_simulation=False and populate the env vars above to use a real environment
dv_client = DataverseClient(use_simulation=True)
print("DataverseClient initialised (simulation mode)")

DataverseClient initialised (simulation mode)


In [17]:
# Full Pipeline: LangChain Analysis -> Dataverse Record Creation
#
# This pipeline takes a support request, analyses it using LangChain,
# and writes the structured result to a Dataverse table.
# In production, this would be triggered by an inbound email to Exchange,
# a Power Automate flow, or a Copilot Studio action.

from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

class SupportTicketAnalysis(BaseModel):
    """Schema for AI-analysed support tickets written to Dataverse."""
    cr_title:         str                                   = Field(description="Short ticket title, max 100 chars")
    cr_category:      Literal["billing","technical","access","general"] = Field(description="Ticket category")
    cr_priority:      Literal["critical","high","medium","low"]         = Field(description="Ticket priority")
    cr_summary:       str                                   = Field(description="Two-sentence issue summary")
    cr_suggested_team:str                                   = Field(description="Team that should handle this")
    cr_ai_confidence: float                                 = Field(description="AI confidence score 0.0 to 1.0")

llm = get_llm(temperature=0.0)
structured_llm = llm.with_structured_output(SupportTicketAnalysis)

ticket_analysis_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "You are an enterprise support triage AI. Analyse the support request and "
         "produce a structured record for the Dataverse ticketing system. "
         "Be specific with team routing: use team names like "
         "'Azure Infrastructure', 'Identity and Access', 'Application Support', 'Finance Ops'."),
        ("human", "{support_request}")
    ]) | structured_llm
)

def process_and_store_ticket(support_request: str) -> dict:
    """
    Analyse a support request with LangChain and write the result to Dataverse.
    Returns the created Dataverse record.
    """
    # Step 1: Analyse with LangChain
    analysis: SupportTicketAnalysis = ticket_analysis_chain.invoke({
        "support_request": support_request
    })

    # Step 2: Map Pydantic model to Dataverse record dict
    # Dataverse column names use the publisher prefix (cr_ here)
    record = {
        "cr_title":          analysis.cr_title,
        "cr_category":       analysis.cr_category,
        "cr_priority":       analysis.cr_priority,
        "cr_summary":        analysis.cr_summary,
        "cr_suggested_team": analysis.cr_suggested_team,
        "cr_ai_confidence":  analysis.cr_ai_confidence,
        "cr_raw_request":    support_request[:2000],
        "cr_ai_processed":   True,
    }

    # Step 3: Write to Dataverse
    # Table name: cr_supporttickets (your custom table)
    created = dv_client.create_record("cr_supporttickets", record)
    return created

# Test with a realistic support request
test_request = """
From: Sarah Mitchell, Senior Finance Analyst
Subject: Cannot access Azure cost reports since password reset

Since I reset my password this morning following the enforced rotation policy,
I can no longer access the Azure Cost Management portal. I am getting an error:
'You do not have permission to view billing information for this subscription.'
I need access urgently as the monthly close report is due by end of day.
My manager is David Chen (david.chen@company.com).
"""

print("PROCESSING SUPPORT REQUEST")
print("=" * 60)
created_record = process_and_store_ticket(test_request)

PROCESSING SUPPORT REQUEST
  [SIMULATION] Created record in cr_supporttickets:
  {
  "cr_title": "Access issue with Azure Cost Management portal",
  "cr_category": "access",
  "cr_priority": "high",
  "cr_summary": "The user is unable to access the Azure Cost Management portal after a password reset, receiving a permission error. This issue is urgent due to a pending monthly close report deadline.",
  "cr_suggested_team": "Identity and Access",
  "cr_ai_confidence": 0.95,
  "cr_raw_request": "\nFrom: Sarah Mitchell, Senior Finance Analyst\nSubject: Cannot access Azure cost reports since password reset\n\nSince I reset my password this morning following the enforced rotation policy,\nI can no longer access the Azure Cost Management portal. I am getting an error:\n'You do not have permission to view billing information for this subscription.'\nI need access urgently as the monthly close report is due by end of day.\nMy manager is David Chen (david.chen@company.com).\n",
  "cr_ai_processe

c:\Python\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SupportTicketAnalysis(cr_..., cr_ai_confidence=0.95), input_type=SupportTicketAnalysis])
  return self.__pydantic_serializer__.to_python(
C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10488\1485751012.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  simulated = {**record, "cr_id": str(uuid.uuid4()), "createdon": datetime.utcnow().isoformat()}


### Power Automate Integration Pattern

The pattern above runs from Python. In a no-code enterprise deployment, the same flow is orchestrated by Power Automate. A Power Automate cloud flow triggers on a new email arriving in a shared mailbox (using the Office 365 connector), calls your LangChain endpoint (using the HTTP connector), and writes the response to Dataverse (using the Dataverse connector). No Python is executed by the business user; they only see the resulting ticket in their Power Apps interface.

To connect Copilot Studio to this data, you add a Dataverse knowledge source to your agent. The agent can then answer questions like "show me open P1 tickets assigned to Azure Infrastructure" by querying the Dataverse table directly, with no additional code required.

The full architecture looks like this:

```
Inbound Email
     |
     v
Power Automate (trigger: new email to shared mailbox)
     |
     v
HTTP Action -> LangChain Pipeline (Azure Function / Container App)
     |
     v
Dataverse connector -> Write to cr_supporttickets table
     |
     v
Power Apps (ticket management UI for support team)
Copilot Studio (Teams bot for managers to query ticket status)
Power BI (real-time dashboard of AI triage accuracy and volume)
```

## Section 7: End-to-End Orchestration Pipeline

### Concept

This final section assembles the patterns from all previous sections into a single cohesive pipeline. The pipeline demonstrates how orchestration, reasoning, validation, and branching work together in a realistic enterprise scenario: processing inbound business documents, routing them to the right analysis path, validating the outputs, and writing results to a structured store that downstream systems (Copilot Studio, Power Apps) can query.

The scenario is a corporate document triage system. Documents arrive from various sources (email, SharePoint, Teams). The pipeline classifies each document, applies the appropriate reasoning strategy for its type, validates the output, and writes the structured result to Dataverse. A Copilot Studio agent allows business users to query the processed documents in natural language.

In [18]:
# End-to-End Pipeline: Document Triage, Reasoning, Validation, Storage

from pydantic import BaseModel, Field
from typing import Literal, List, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnablePassthrough, RunnableLambda

# Output schema for all processed documents
class ProcessedDocument(BaseModel):
    doc_type:        Literal["contract","incident","proposal","compliance","general"]
    title:           str            = Field(description="Short descriptive title")
    analysis:        str            = Field(description="Full analysis output")
    recommended_action: str         = Field(description="Single concrete recommended action")
    urgency:         Literal["immediate","this-week","this-month","no-action"]
    route_to:        str            = Field(description="Team or person to route this to")
    validation_passed: bool         = Field(description="Whether validation checks passed")
    confidence:      float          = Field(description="0.0 to 1.0")

classifier_llm = get_llm(temperature=0.0)
analyst_llm    = get_llm(temperature=0.2, max_tokens=700)
validator_llm  = get_llm(temperature=0.0, max_tokens=200)
str_out        = StrOutputParser()

# Stage 1: Classify
classify = (
    ChatPromptTemplate.from_template(
        "Classify into one category: contract/incident/proposal/compliance/general.\n"
        "Reply with just the category label.\n\nDocument:\n{document}"
    ) | classifier_llm | str_out
)

# Stage 2: Type-specific analysis chains
contract_analysis = (
    ChatPromptTemplate.from_template(
        "Analyse this contract. Reason step by step through: parties, obligations, "
        "risks, value, and recommendation.\n\n{document}"
    ) | analyst_llm | str_out
)
incident_analysis = (
    ChatPromptTemplate.from_template(
        "Analyse this incident report. Cover: severity, root cause hypothesis, "
        "immediate actions, and notification requirements.\n\n{document}"
    ) | analyst_llm | str_out
)
proposal_analysis = (
    ChatPromptTemplate.from_template(
        "Analyse this proposal against Well-Architected principles. "
        "Cover each pillar briefly. Give a verdict.\n\n{document}"
    ) | analyst_llm | str_out
)
compliance_analysis = (
    ChatPromptTemplate.from_template(
        "Analyse this compliance document. Identify: regulations referenced, "
        "gaps, controls required, and timeline.\n\n{document}"
    ) | analyst_llm | str_out
)
general_analysis = (
    ChatPromptTemplate.from_template(
        "Summarise this document and identify any actions required.\n\n{document}"
    ) | analyst_llm | str_out
)

analysis_branch = RunnableBranch(
    (lambda x: "contract"   in x["doc_type"], contract_analysis),
    (lambda x: "incident"   in x["doc_type"], incident_analysis),
    (lambda x: "proposal"   in x["doc_type"], proposal_analysis),
    (lambda x: "compliance" in x["doc_type"], compliance_analysis),
    general_analysis,
)

# Stage 3: Structured output from analysis
structured_output_llm = analyst_llm.with_structured_output(ProcessedDocument)

structure_result = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Convert the analysis below into a structured document record. "
         "Be precise with urgency and routing. "
         "Set validation_passed=True only if the analysis is specific and actionable."),
        ("human",
         "Document type: {doc_type}\n\nOriginal document:\n{document}\n\nAnalysis:\n{analysis}")
    ]) | structured_output_llm
)

# Assemble the full pipeline
full_triage_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(doc_type=classify)
    | RunnablePassthrough.assign(
        _log=lambda x: print(f"  Document classified as: {x['doc_type'].strip()}") or ""
    )
    | RunnablePassthrough.assign(analysis=analysis_branch)
    | RunnablePassthrough.assign(structured=structure_result)
)

# Run the pipeline on three document types
final_test_docs = [
    """
    MASTER SERVICE AGREEMENT: TechPartner Solutions will provide cloud consulting
    services for 12 months at USD 25,000 per month. Scope: Azure migration advisory.
    IP created during engagement remains with TechPartner unless separately agreed.
    Limitation of liability: 1x monthly fee. Governed by English law.
    """,
    """
    GDPR DATA PROCESSING AUDIT FINDINGS: Customer data is currently replicated to
    Azure East US region as part of disaster recovery. Data subjects are EU residents.
    No Standard Contractual Clauses (SCCs) in place for US transfer. DPA not signed
    with third-party analytics vendor receiving anonymised event data.
    """,
]

for doc in final_test_docs:
    print("\n" + "=" * 70)
    result = full_triage_pipeline.invoke(doc)
    pd_result: ProcessedDocument = result["structured"]

    print(f"Title              : {pd_result.title}")
    print(f"Type               : {pd_result.doc_type}")
    print(f"Urgency            : {pd_result.urgency}")
    print(f"Route To           : {pd_result.route_to}")
    print(f"Recommended Action : {pd_result.recommended_action}")
    print(f"Validation Passed  : {pd_result.validation_passed}")
    print(f"Confidence         : {pd_result.confidence:.0%}")
    print(f"\nAnalysis Summary:")
    print(pd_result.analysis[:400] + "...")

    # Write to Dataverse
    dv_client.create_record("cr_processeddocuments", pd_result.dict())


  Document classified as: contract


c:\Python\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ProcessedDocument(doc_typ...d=True, confidence=0.95), input_type=ProcessedDocument])
  return self.__pydantic_serializer__.to_python(
C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10488\1145300939.py:129: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  dv_client.create_record("cr_processeddocuments", pd_result.dict())
C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_10488\1485751012.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
 

Title              : Analysis of Master Service Agreement
Type               : contract
Urgency            : this-week
Route To           : Legal Team
Recommended Action : Negotiate IP ownership terms and clarify scope of work.
Validation Passed  : True
Confidence         : 95%

Analysis Summary:
The contract outlines the terms of engagement between TechPartner Solutions and the client for cloud consulting services, focusing on Azure migration advisory. Key points include the service duration, monthly fee, IP ownership, liability limitations, and governing law. Risks and benefits for both parties are identified, with recommendations for negotiation on IP ownership, scope clarity, and liabi...
  [SIMULATION] Created record in cr_processeddocuments:
  {
  "doc_type": "contract",
  "title": "Analysis of Master Service Agreement",
  "analysis": "The contract outlines the terms of engagement between TechPartner Solutions and the client for cloud consulting services, focusing on Azure migrat

## Summary and Next Steps

This lab covered the core orchestration patterns for enterprise AI systems using Azure OpenAI and LangChain.

**Workflow Orchestration** taught you how to sequence, parallelise, and batch LLM calls using LCEL. The key takeaway is that every step in a pipeline is a Runnable, and `RunnablePassthrough.assign` is how you accumulate state across steps without losing prior outputs.

**Intermediate Reasoning** showed that how you prompt matters as much as what you prompt. Chain-of-Thought produces better reasoning on complex problems. ReAct connects reasoning to real data via tools. Self-reflection adds a quality gate that catches errors before they propagate.

**Validation Prompts** established that LLM outputs must be validated at boundary points. Pydantic handles structural validation. A second LLM call handles semantic validation. Auto-retry with enriched prompts closes the loop when validation fails.

**Branching** showed how to build pipelines that handle heterogeneous inputs by classifying first and routing second. Multi-level branching handles hierarchical categories. Each branch can be a full sub-pipeline.

**Copilot Studio and Power Platform** connected the engineering layer to the business layer. Your pipelines become HTTP endpoints that Copilot Studio calls as actions. Dataverse stores structured AI outputs. Power Automate orchestrates the data flow. Power Apps and Copilot Studio surface it to users.

**Recommended next steps for production deployment:**

Deploy your pipelines to Azure Container Apps or Azure Functions for scalable, managed hosting. Add Azure Application Insights callbacks (as shown in the Long LangChains lab) to every production pipeline for observability. Use Azure API Management as the gateway in front of your endpoints when connecting to Copilot Studio, to handle authentication, rate limiting, and request logging centrally. Store conversation history in Azure Cosmos DB using `CosmosDBChatMessageHistory` for durable multi-turn memory across sessions. Enable LangSmith tracing during development to trace every step of complex pipelines.